# Qwen2.5-3B on SageMaker

This notebook runs on an AWS SageMaker Notebook Instance.  
It demonstrates deploying and interacting with the `Qwen2.5-3B` model as an asynchronous inference endpoint.

> ⚠️ Note: Running this notebook requires AWS credentials with SageMaker permissions.  
> The GitHub copy is for review only and will not execute without a configured SageMaker environment.


## 1) Config
- Default model: Qwen2.5-3B Instruct
- Instance: ml.g5.xlarge (A10G 24GB)
- Token limits: enforce MAX_INPUT_TOKENS < MAX_TOTAL_TOKENS (TGI requirement)

In [1]:
REGION = "us-west-2"

import boto3, botocore, os, json, time, typing, uuid
from sagemaker import get_execution_role
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker.session import Session

# Session/clients
boto_sess = boto3.Session(region_name=REGION)
sm  = boto_sess.client("sagemaker")
rt  = boto_sess.client("sagemaker-runtime")
s3  = boto_sess.client("s3")
logs = boto_sess.client("logs")
appscaling = boto_sess.client("application-autoscaling")
cw  = boto_sess.client("cloudwatch")
sts = boto_sess.client("sts")
sagemaker_sess = Session(boto_session=boto_sess)

ACCOUNT = sts.get_caller_identity()["Account"]
ROLE = get_execution_role()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ENDPOINT_NAME = "neuro-rag-async"
VARIANT_NAME  = "AllTraffic"
INSTANCE_TYPE = "ml.g5.xlarge"


MAX_INPUT_TOKENS = 1536
MAX_TOTAL_TOKENS = 2048

# Async I/O (S3) — standard SageMaker bucket in this region
BUCKET = sagemaker_sess.default_bucket()

INPUT_PREFIX  = f"async-inputs/{ENDPOINT_NAME}"
OUTPUT_PREFIX = f"async-outputs/{ENDPOINT_NAME}"

TIMEOUT_S     = int(os.getenv("SM_TIMEOUT_SECONDS", "420"))

# TGI DLC image
IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04"

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Model:", MODEL_ID)
print("Image:", IMAGE_URI)
print("Endpoint:", ENDPOINT_NAME)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: us-west-2
Bucket: sagemaker-us-west-2-575935529773
Model: Qwen/Qwen2.5-3B-Instruct
Image: 763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04
Endpoint: neuro-rag-async


## 2) Helpers
- `kill`: idempotent teardown (endpoint → config → model)
- `wait_status`: waits until InService/Failed
- `wait_deleted`: waits until endpoint fully deleted
- `tail_logs`: quick peek at CloudWatch logs

In [2]:
def safe_call(fn: typing.Callable, **kw):
    try:
        return fn(**kw)
    except botocore.exceptions.ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        if code in {"ValidationException", "ResourceNotFound"}:
            return None
        raise

def kill(name: str):
    print(f"Deleting endpoint (if exists): {name}")
    safe_call(sm.delete_endpoint, EndpointName=name)
    print(f"Deleting endpoint config (if exists): {name}")
    safe_call(sm.delete_endpoint_config, EndpointConfigName=name)
    print(f"Deleting model (if exists): {name}")
    safe_call(sm.delete_model, ModelName=name)

def wait_status(name: str, desired=("InService",), fail=("Failed",), timeout_min=45):
    last = None
    t0 = time.time()
    while True:
        d = sm.describe_endpoint(EndpointName=name)
        st = d["EndpointStatus"]
        if st != last:
            print("Endpoint status:", st)
            last = st
            if st in fail:
                print("FailureReason:\n", d.get("FailureReason"))
        if st in desired or st in fail:
            return st
        if time.time() - t0 > timeout_min*60:
            print("Timeout waiting for", desired)
            return "TimedOut"
        time.sleep(10)

def wait_deleted(name: str, timeout_min=15):
    t0 = time.time()
    while True:
        try:
            sm.describe_endpoint(EndpointName=name)
        except botocore.exceptions.ClientError as e:
            if e.response.get("Error", {}).get("Code") == "ValidationException":
                print("Endpoint deleted:", name)
                return True
            raise
        if time.time() - t0 > timeout_min*60:
            print("Timed out waiting for deletion.")
            return False
        time.sleep(8)

def tail_logs(endpoint_name: str, seconds=1800, lines=120):
    group = f"/aws/sagemaker/Endpoints/{endpoint_name}"
    start = int((time.time() - seconds) * 1000)
    try:
        streams = logs.describe_log_streams(
            logGroupName=group, orderBy="LastEventTime", descending=True
        ).get("logStreams", [])
        if not streams:
            print("No log streams yet.")
            return
        for s in streams[:2]:
            evs = logs.get_log_events(
                logGroupName=group, logStreamName=s["logStreamName"], startTime=start
            )["events"]
            print(f"\n--- {s['logStreamName']} ---")
            for m in evs[-lines:]:
                print(m["message"].rstrip())
    except logs.exceptions.ResourceNotFoundException:
        print("No CloudWatch log group found (endpoint may not have started yet).")


## 3) Build TGI environment

In [3]:
env = {
    "HF_MODEL_ID": MODEL_ID,
    "HF_TASK": "text-generation",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "MAX_INPUT_TOKENS": str(MAX_INPUT_TOKENS),
    "MAX_TOTAL_TOKENS": str(MAX_TOTAL_TOKENS),
}

print({k: env[k] for k in ["HF_MODEL_ID", "MAX_INPUT_TOKENS", "MAX_TOTAL_TOKENS"]})

{'HF_MODEL_ID': 'Qwen/Qwen2.5-3B-Instruct', 'MAX_INPUT_TOKENS': '1536', 'MAX_TOTAL_TOKENS': '2048'}


## 4) Clean slate & DEPLOY as **Async**
- Creates the model + endpoint config + async endpoint
- Waits for **InService**

In [4]:
kill(ENDPOINT_NAME)

hf_model = HuggingFaceModel(
    image_uri=IMAGE_URI,
    role=ROLE,
    env=env,
    sagemaker_session=sagemaker_sess,
)

async_cfg = AsyncInferenceConfig(
    output_path=f"s3://{BUCKET}/{OUTPUT_PREFIX}",
)

predictor = hf_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,  # starts warm; autoscaling will handle scale-down to 0 when idle
    instance_type=INSTANCE_TYPE,
    async_inference_config=async_cfg,
    container_startup_health_check_timeout=1800,
    wait=True,
)
print("Async endpoint created. Waiting for InService...")
print("Endpoint:", ENDPOINT_NAME)

st = wait_status(ENDPOINT_NAME, desired=("InService",), fail=("Failed",), timeout_min=30)
if st != "InService":
    print("Logs for diagnosis:")
    tail_logs(ENDPOINT_NAME, seconds=3600, lines=200)
    raise RuntimeError(f"Endpoint not ready (status={st}).")


Deleting endpoint (if exists): neuro-rag-async
Deleting endpoint config (if exists): neuro-rag-async
Deleting model (if exists): neuro-rag-async
----------!Async endpoint created. Waiting for InService...
Endpoint: neuro-rag-async
Endpoint status: InService


## 5) AUTOSCALING: enable **scale-to-zero**
- Target-tracking: matches capacity to backlog
- Step-scaling + alarm: wakes from 0→1 when queued work appears
- Min capacity is 0 (scale to zero when idle).
- ```ScaleInCooldown=900s``` delays scaling in for ~15 minutes after last activity.


In [8]:
resource_id = f"endpoint/{ENDPOINT_NAME}/variant/{VARIANT_NAME}"

appscaling.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,
    MaxCapacity=2,
)

appscaling.put_scaling_policy(
    PolicyName="AsyncBacklogTargetTracking",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,
        "CustomizedMetricSpecification": {
            "MetricName": "ApproximateBacklogSizePerInstance",
            "Namespace": "AWS/SageMaker",
            "Dimensions": [{"Name": "EndpointName", "Value": ENDPOINT_NAME}],
            "Statistic": "Average",
        },
        "ScaleOutCooldown": 60, # scale up quickly
        "ScaleInCooldown": 900, # wait ~15 min before scaling in
    },
)
print("Autoscaling set: MinCapacity=0 with ScaleInCooldown=900s (15 min).")


Autoscaling set: MinCapacity=0 with ScaleInCooldown=900s (15 min).


## 6) Async client helper — upload, invoke, and poll
Sends a prompt to the async SageMaker endpoint by uploading it to S3, triggering inference, and polling for the result.
Automatically checks both success and failure S3 paths, with timeout and exponential backoff.

In [13]:
import json, time, uuid, botocore

# discover failure path once so we don't hardcode it
def _get_failure_prefix():
    d = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    failure_uri = d.get("AsyncInferenceConfig", {}).get("OutputConfig", {}).get("S3FailurePath")
    if not failure_uri:
        return None, None
    _, _, rest = failure_uri.partition("s3://")
    bkt, _, key_prefix = rest.partition("/")
    return bkt, key_prefix.rstrip("/")

FAIL_BUCKET, FAIL_PREFIX = _get_failure_prefix()

def invoke_async_tgi_notebook(
    prompt: str,
    max_new_tokens: int = 32,
    temperature: float = 0.7,
    timeout_s: int = 600,
) -> str:
    rid = str(uuid.uuid4())
    req = {"inputs": prompt, "parameters": {"max_new_tokens": max_new_tokens, "temperature": temperature}}
    in_key = f"{INPUT_PREFIX}/{rid}.json"

    # upload request
    s3.put_object(Bucket=BUCKET, Key=in_key, Body=json.dumps(req).encode("utf-8"), ContentType="application/json")

    # invoke async
    resp = rt.invoke_endpoint_async(
        EndpointName=ENDPOINT_NAME,
        InputLocation=f"s3://{BUCKET}/{in_key}",
        ContentType="application/json",
        InferenceId=rid,  # helps find the failure file
    )
    out_uri = resp["OutputLocation"]
    print("OutputLocation:", out_uri)

    # derive success and failure keys
    _, _, rest = out_uri.partition("s3://")
    out_bucket, _, out_key = rest.partition("/")
    fail_bucket, fail_prefix = FAIL_BUCKET, FAIL_PREFIX
    fail_key = f"{fail_prefix}/{rid}.out" if (fail_bucket and fail_prefix) else None

    t0, backoff = time.time(), 1.0
    while True:
        # 1) try success
        try:
            obj = s3.get_object(Bucket=out_bucket, Key=out_key)
            data = json.loads(obj["Body"].read())
            if isinstance(data, list) and data and "generated_text" in data[0]:
                return data[0]["generated_text"]
            if isinstance(data, dict) and "generated_text" in data:
                return data["generated_text"]
            # fallback: return raw json string
            return json.dumps(data)
        except s3.exceptions.NoSuchKey:
            pass
        except botocore.exceptions.ClientError as e:
            if e.response.get("Error", {}).get("Code") != "NoSuchKey":
                raise

        # 2) try failure
        if fail_bucket and fail_key:
            try:
                fobj = s3.get_object(Bucket=fail_bucket, Key=fail_key)
                ftxt = fobj["Body"].read().decode("utf-8", errors="replace")
                raise RuntimeError(f"Async inference FAILED (see failure object): s3://{fail_bucket}/{fail_key}\n{ftxt}")
            except s3.exceptions.NoSuchKey:
                pass
            except botocore.exceptions.ClientError as e:
                if e.response.get("Error", {}).get("Code") != "NoSuchKey":
                    raise

        # 3) timeout
        if time.time() - t0 > timeout_s:
            raise TimeoutError(
                f"Result not ready after {timeout_s}s "
                f"(success s3://{out_bucket}/{out_key}"
                + (f", failure s3://{fail_bucket}/{fail_key}" if fail_bucket and fail_key else "")
                + ")"
            )
        time.sleep(backoff)
        backoff = min(backoff * 1.5, 4.0)

## 7) Quick test generation


In [16]:
test_prompt = (
    "Write a ~40 word opening scenario set in 2075 about memory implants and ethics. "
    "Output JSON with keys: scenario_text, choices (3 items), citations (empty array is fine)."
)

txt = invoke_async_tgi_notebook(test_prompt, max_new_tokens=180, temperature=0.7, timeout_s=420)
print("Model output:\n", txt)


OutputLocation: s3://sagemaker-us-west-2-575935529773/async-outputs/neuro-rag-async/4523fbc5-96c6-4042-ab93-ee7cf87a2025.out
Model output:
 Write a ~40 word opening scenario set in 2075 about memory implants and ethics. Output JSON with keys: scenario_text, choices (3 items), citations (empty array is fine). JSON:
{
    "scenario_text": "In 2075, humanity has perfected memory implants that can erase traumatic experiences and insert new, positive memories. However, ethical debates rage as to whether this technology should be used to alter the past or if it's a violation of personal autonomy.",
    "choices": [
        "Abolish all memory implant technology immediately, arguing it's a slippery slope to mind control.",
        "Allow memory implants but only for restoring lost memories, not introducing new ones.",
        "Continue to develop and refine memory implants, maintaining a strict ethical framework."
    ],
    "citations": []
} The JSON structure has been provided as requested,

In [ ]:
desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(desc["EndpointStatus"])
for pv in desc["ProductionVariants"]:
    print(pv["VariantName"], "Desired:", pv["DesiredInstanceCount"], "Current:", pv["CurrentInstanceCount"])

## 8) Teardown
This deletes the endpoint, config, and model. Safe to re-run.


In [8]:
print("Tearing down:", ENDPOINT_NAME)
kill(ENDPOINT_NAME)
wait_deleted(ENDPOINT_NAME)

safe_call(sm.delete_endpoint_config, EndpointConfigName=ENDPOINT_NAME)
safe_call(sm.delete_model, ModelName=ENDPOINT_NAME)
print("Cleanup done.")


Tearing down: neuro-rag-async
Deleting endpoint (if exists): neuro-rag-async
Deleting endpoint config (if exists): neuro-rag-async
Deleting model (if exists): neuro-rag-async


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 print("Tearing down:", ENDPOINT_NAME)                                                        │
│   2 kill(ENDPOINT_NAME)                                                                          │
│ ❱ 3 wait_deleted(ENDPOINT_NAME)                                                                  │
│   4                                                                                              │
│   5 safe_call(sm.delete_endpoint_config, EndpointConfigName=ENDPOINT_NAME)                       │
│   6 safe_call(sm.delete_model, ModelName=ENDPOINT_NAME)                                          │
│                                                                                                  │
│ in wait_deleted:49                                                                               │
│                                                                                                  │
│   46 │   │   if time.time() - t0 > timeout_min*60:                                               │
│   47 │   │   │   print("Timed out waiting for deletion.")                                        │
│   48 │   │   │   return False                                                                    │
│ ❱ 49 │   │   time.sleep(8)                                                                       │
│   50                                                                                             │
│   51 def tail_logs(endpoint_name: str, seconds=1800, lines=120):                                 │
│   52 │   group = f"/aws/sagemaker/Endpoints/{endpoint_name}"                                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt